# Day 039 · 数据合并 merge/concat · 中国版
**Joins** · 阶段 P2 · Python 量化工具栈

> 选股常常要把两类数据拼起来:行情和财务、价格和因子、股票和行业标签。这一节用真实 A 股的日线加季度财报,带你把 concat 沿轴拼接、merge 按键对齐(类 SQL JOIN)、merge_asof 时间就近对齐这三件套讲透,重点是量化里最实用、也最容易出错的一步——把财报安全地对齐到每个交易日,绝不偷看未来。

---

### 关于「中国版」

本 notebook 是为**国内学员**优化的版本:
- 数据源用 **akshare**(国内可访问、零 VPN、免注册),取代了视频里的 yfinance
- 标的尽量保持原意:美股 ETF→A 股 ETF / 国际公司→A 股龙头
- 所讲的**概念和方法 100% 一致**,但**具体数字可能与视频里略有差异**(因为是不同时间窗 / 不同标的)
- 一般情况国内 `pip install akshare` 即可,无需 token / VPN

**课件生成日期:** 2026-05-30  ·  **建议学习时长:** 20 分钟

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有必需的 Python 包(含 `akshare`),缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续


In [ ]:
# === 环境自检 + 自动安装(运行此单元格即可)===
import importlib, subprocess, sys, os

REQUIRED = ["akshare", "baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels"]
PIP_NAME = {"sklearn":"scikit-learn","cv2":"opencv-python","PIL":"Pillow","bs4":"beautifulsoup4","yaml":"PyYAML"}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))
if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置 ===
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm
CJK = ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
       "C:/Windows/Fonts/msyh.ttc","C:/Windows/Fonts/simhei.ttf",
       "/System/Library/Fonts/PingFang.ttc","/System/Library/Fonts/STHeiti Medium.ttc"]
for p in CJK:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP","Microsoft YaHei","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪")


## 🔌 第二步:加载国内数据助手

下面这一格是**工具函数**(可以折叠,不需要修改)。它把 `yfinance` 风格的 ticker(如 `600519.SS`)自动路由到对应的 akshare 接口,提供 `get_close(ticker)` 和 `get_close_multi(tickers_dict)` 两个函数。

In [ ]:
# === 国内数据源助手(akshare 后端,不需要 VPN)===
# 这一格是工具函数,可以折叠,不需要修改。
# 它把 yfinance 风格的 ticker(如 "600519.SS" / "0700.HK" / "AAPL" / "BTC-USD")
# 自动路由到对应的 akshare 接口,统一返回 yfinance 风格的 Close DataFrame。

import re
from datetime import datetime, timedelta
import pandas as pd
import akshare as ak

_TICKER_MAP = {
    "^GSPC": ("us_index_sina", ".INX"),
    "^DJI":  ("us_index_sina", ".DJI"),
    "^IXIC": ("us_index_sina", ".IXIC"),
    "GC=F":  ("foreign_futures", "GC"),
    "SI=F":  ("foreign_futures", "SI"),
    "CL=F":  ("foreign_futures", "CL"),
    "BTC-USD": ("crypto", "BTC"),
    "ETH-USD": ("crypto", "ETH"),
}

def _retry(fn, *args, _retries=4, _wait=1.5, **kwargs):
    """akshare 上游(东方财富/新浪/Binance)偶有 RemoteDisconnected / Timeout,自动重试 4 次。
    2026-05-11 加:用户跑 cn notebook 拉 002594.SZ 时上游断连 → 整节卡死。
    每次重试间隔 _wait 秒(指数退避 1x → 1.5x → 2.25x)。
    """
    import time as _t
    last_err = None
    wait = _wait
    for i in range(_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            name = type(e).__name__
            if i == _retries - 1:
                print(f"  ✗ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次仍失败({name}),放弃")
                raise
            print(f"  ⚠ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次失败({name}),{wait:.1f}s 后重试")
            _t.sleep(wait)
            wait *= 1.5

def _parse_period(period):
    end = datetime.today()
    m = re.match(r"^(\d+)\s*(y|mo|d|w)$", period.lower().strip())
    days = 365 * 3 if not m else int(m.group(1)) * {"y":365,"mo":30,"w":7,"d":1}[m.group(2)]
    return (end - timedelta(days=days+30)).strftime("%Y%m%d"), end.strftime("%Y%m%d")

def _classify(ticker):
    t = ticker.strip()
    if t in _TICKER_MAP: return _TICKER_MAP[t]
    if t.endswith((".SS",".SH",".SZ")):
        code = t.split(".")[0]
        if code.startswith(("51","159","58")) or code in ("510300","510500","510050","511010","513100"):
            return ("a_etf", code)
        if code in ("000300","000016","000905","000852","000001"):
            return ("a_index", code)
        return ("a_stock", code)
    if t.endswith(".HK"):
        return ("hk", t.split(".")[0].zfill(5))
    return ("us", t)

def _norm(df, dc, cc):
    out = df[[dc, cc]].copy()
    out[dc] = pd.to_datetime(out[dc])
    return out.set_index(dc).sort_index()[cc].astype(float).rename("Close")

def get_close(ticker, period="3y"):
    """返回某标的 Close 价格 series。后端 akshare,中国可访问。
    所有 ak.* 调用都过 _retry(4 次,指数退避)— 防东方财富/新浪上游瞬时断连。
    """
    start, end = _parse_period(period)
    kind, sym = _classify(ticker)
    if kind == "a_stock":
        return _norm(_retry(ak.stock_zh_a_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_etf":
        return _norm(_retry(ak.fund_etf_hist_em, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_index":
        idx_map = {"000300":"sh000300","000016":"sh000016","000905":"sh000905","000852":"sh000852","000001":"sh000001"}
        s = _norm(_retry(ak.stock_zh_index_daily_em, symbol=idx_map.get(sym, f"sh{sym}")), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "hk":
        return _norm(_retry(ak.stock_hk_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "us":
        s = _norm(_retry(ak.stock_us_daily, symbol=sym, adjust="qfq"), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "us_index_sina":
        s = _norm(_retry(ak.index_us_stock_sina, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "foreign_futures":
        s = _norm(_retry(ak.futures_foreign_hist, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "crypto":
        import requests as _rq
        def _binance():
            r = _rq.get("https://api.binance.com/api/v3/klines",
                        params={"symbol": f"{sym}USDT", "interval": "1d", "limit": 1000}, timeout=15)
            r.raise_for_status()
            return r.json()
        klines = _retry(_binance)
        df = pd.DataFrame(klines, columns=["open_time","open","high","low","close","volume",
                                            "close_time","qav","trades","tbb","tbq","ignore"])
        df["date"] = pd.to_datetime(df["open_time"], unit="ms")
        df["close"] = df["close"].astype(float)
        s = df.set_index("date").sort_index()["close"].rename("Close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    raise ValueError(f"unsupported ticker: {ticker}")

def get_close_multi(tickers, period="3y"):
    """批量取 Close,返回 DataFrame,列名是 tickers dict 的 key(中文名),按交集日期对齐。"""
    series = {name: get_close(t, period=period) for name, t in tickers.items()}
    return pd.concat(series, axis=1).sort_index()

print("✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据")


## 学习目标

- 理解 concat 沿轴拼接:把多只票的数据竖着叠、或多列横着接
- 掌握 merge 按键对齐,理解 inner 和 outer join 的区别
- 看懂 outer join 产生的 NaN 是"没匹配上"的信号,不是数据本身缺
- 会用 merge_asof 把低频财报按时间就近对齐到高频行情,杜绝未来函数
- 能把行情和财务拼成一张表,做出一个简单的按 ROE 选股

## 历史背景:小孙把财报和股价按行号一拼,选股全错

小孙想做个基本面选股:谁的净资产收益率高就买谁。他下载了股价的日线表和财报的季度表,想当然地把两张表按行号直接拼在一起。结果灾难发生了:股价是每个交易日一行,一年两百多行;财报是每季一行,一年才四行。按行号硬拼,等于让3 月一号的股价去配第三季度的财报,时间完全错位。更隐蔽的是,他用了当年最新的年报数据去回测年初的行情,相当于拿着还没公布的财报去买年初的股票,这就是典型的偷看未来,回测好得离谱,实盘一塌糊涂。这一节我们就用 merge_asof,教你把每个交易日只对齐到那天之前最近一期已经披露的财报,既拼得上,又绝不偷看未来。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. concat 沿轴拼接

就好比把几张同样格式的表上下摞起来(竖着叠,增加行),或者左右并起来(横着接,增加列)。把十只票各自的日线竖着叠成一张大长表,就是 concat。


### 2. merge 按键对齐

等于说就是数据库里的 JOIN:指定一个共同的键(比如股票代码),把两张表按这个键对齐拼到一起。两张 Excel 用学号对齐成绩和身高,就是 merge。


### 3. inner 与 outer join

inner 只保留两张表都有的键,outer 保留并集。outer join 里没对上的地方会变成 NaN,这个 NaN 是"这条没匹配上"的信号,提醒你两张表对不齐,而不是数据本身缺了。


### 4. merge_asof 就近对齐

专门给时间序列用的"就近匹配"。它不要求键完全相等,而是为左表每一行找右表里时间最接近的一行。设成 backward,就只匹配当天或之前的记录,天然杜绝偷看未来。


### 5. point-in-time 时点对齐

回测的铁律:某一天只能用那一天真实能看到的信息。财报有披露滞后,merge_asof backward 保证每个交易日只对齐到已经公布的财报,这叫时点正确。


## 实操:数据合并五件套 — concat 拼接 / merge 类 JOIN / outer 的 NaN / merge_asof 时序就近对齐 / 行情+财务选股

下面这段代码用 akshare 抓数据,国内零 VPN 跑通。**直接 Run All** 看结果。

**依赖:** `pip install pandas numpy matplotlib akshare statsmodels scipy`

In [ ]:
# day_039_merge.py — 数据合并:merge(类 SQL JOIN)/ concat 拼接 / merge_asof 时序就近对齐 / outer join 的 NaN
# 把行情(日线)和财务(季度 ROE/EPS)对齐起来选股,讲透最容易出错的"日期对不上"
# 数据:baostock 日线 + 季频财务(免费、稳定、国内零翻墙)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import baostock as bs

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)

STOCKS = {'sh.600519': '茅台', 'sh.600036': '招商银行', 'sz.000858': '五粮液'}
START, END = '2023-01-01', '2024-12-31'

print('==== 0. 数据拉取(baostock 日线 + 季度财务)====')
lg = bs.login()
if lg.error_code != '0':
    raise RuntimeError(f'baostock 登录失败:{lg.error_msg}')

# 0a 日线收盘
price_frames = {}
for code, name in STOCKS.items():
    rs = bs.query_history_k_data_plus(code, 'date,close', start_date=START, end_date=END,
                                      frequency='d', adjustflag='2')
    rows = []
    while rs.error_code == '0' and rs.next():
        rows.append(rs.get_row_data())
    s = pd.DataFrame(rows, columns=['date', 'close'])
    s['date'] = pd.to_datetime(s['date']); s['close'] = s['close'].astype(float)
    price_frames[name] = s.assign(股票=name)

# 0b 季度盈利能力(ROE / EPS),baostock query_profit_data 按 年+季度
prof_rows = []
for code, name in STOCKS.items():
    for year in (2023, 2024):
        for q in (1, 2, 3, 4):
            rs = bs.query_profit_data(code=code, year=year, quarter=q)
            while rs.error_code == '0' and rs.next():
                r = rs.get_row_data()
                prof_rows.append(r + [name])
    cols = None
# 取一次字段名
rs0 = bs.query_profit_data(code='sh.600519', year=2023, quarter=1)
prof_cols = rs0.fields + ['股票']
bs.logout()

prof = pd.DataFrame(prof_rows, columns=prof_cols)
prof['财报日'] = pd.to_datetime(prof['statDate'])           # 财报截止日
prof['ROE'] = pd.to_numeric(prof['roeAvg'], errors='coerce') * 100
prof['EPS'] = pd.to_numeric(prof['epsTTM'], errors='coerce')
prof = prof[['股票', '财报日', 'ROE', 'EPS']].dropna().sort_values(['股票', '财报日'])
print(f'日线合计 {sum(len(v) for v in price_frames.values())} 行,季度财务 {len(prof)} 行')

# ============ 1. concat:沿轴拼接(把多只票的日线竖着叠起来)============
print('\n==== 1. concat 沿轴拼接 ====')
all_prices = pd.concat(price_frames.values(), axis=0, ignore_index=True)
print(f'concat 把 {len(STOCKS)} 只票的日线竖着叠成一张长表:{all_prices.shape} (行=股票×日期)')
print('每只票行数:', all_prices['股票'].value_counts().to_dict())

# ============ 2. merge:类 SQL JOIN,按键对齐 ============
print('\n==== 2. merge 按键对齐(类 SQL JOIN)====')
# 造一张"行业 + 是否沪深300成分"的小维表,用 merge 贴到行情上
meta = pd.DataFrame({'股票': ['茅台', '招商银行', '五粮液'],
                     '行业': ['白酒', '银行', '白酒'],
                     '入选300': ['是', '是', '是']})
last_price = all_prices.groupby('股票')['close'].last().reset_index()
merged = last_price.merge(meta, on='股票', how='left')        # 按 股票 这个键左连接
print('最新收盘价 merge 行业维表(how=left,on=股票):')
print(merged.to_string(index=False))

# ============ 3. outer join 的 NaN ============
print('\n==== 3. outer join 会暴露不匹配(产生 NaN)====')
meta_missing = meta[meta['股票'] != '五粮液']                # 故意缺五粮液
inner = last_price.merge(meta_missing, on='股票', how='inner')
outer = last_price.merge(meta_missing, on='股票', how='outer')
print(f'inner join(只留两边都有的):{len(inner)} 行,丢掉了维表里没有的五粮液')
print(f'outer join(两边并集):{len(outer)} 行,五粮液的行业/入选300 变成 NaN — 暴露出谁没匹配上')
print('教训:join 前先想清楚要 inner 还是 outer;outer 的 NaN 是"没对上"的信号,不是数据本身缺')

# ============ 4. merge_asof:时间序列就近对齐(行情 + 财务)============
print('\n==== 4. merge_asof 时序就近对齐(point-in-time)====')
# 对每只票:把每个交易日,对齐到"那天能看到的最近一期已披露财报"
out_frames = []
for name in STOCKS.values():
    px = price_frames[name][['date', 'close']].sort_values('date')
    pf = prof[prof['股票'] == name][['财报日', 'ROE', 'EPS']].sort_values('财报日')
    m = pd.merge_asof(px, pf, left_on='date', right_on='财报日', direction='backward')
    m['股票'] = name
    out_frames.append(m)
panel = pd.concat(out_frames, ignore_index=True)
print('merge_asof(direction=backward):每个交易日只对齐到"当天之前"最近披露的财报,杜绝未来函数')
sample = panel[panel['股票'] == '茅台'][['date', 'close', '财报日', 'ROE', 'EPS']]
print('茅台抽样(注意 ROE 随财报日跳变,且财报日 ≤ 交易日):')
print(sample.iloc[[0, 60, 120, 200, 350, -1]].to_string(index=False))

# 用对齐好的 ROE 做个简单选股:每天选 ROE 最高的票
panel_valid = panel.dropna(subset=['ROE'])
pick = panel_valid.loc[panel_valid.groupby('date')['ROE'].idxmax()]
pick_count = pick['股票'].value_counts()
print('\n按"当日可见最新 ROE 最高"选股,各票被选中天数:')
for name, c in pick_count.items():
    print(f'  {name:8s} {c} 天')

# ============ 画图 ============
plt.rcParams['axes.unicode_minus'] = False
# 图1:concat 后各票净值
fig, ax = plt.subplots(figsize=(11, 5.5))
for name in STOCKS.values():
    s = price_frames[name].set_index('date')['close']
    ax.plot(s.index, s / s.iloc[0], label=name, linewidth=1.6)
ax.axhline(1.0, color='gray', ls='--', lw=0.8)
ax.set_title('concat 后三只票净值(起点 1.0)'); ax.legend()
plt.tight_layout(); plt.savefig('chart_01.png', dpi=110); plt.close()

# 图2:茅台 价格 vs 对齐后的 ROE(阶梯)
fig, ax1 = plt.subplots(figsize=(11, 5.5))
mt = panel[panel['股票'] == '茅台'].set_index('date')
ax1.plot(mt.index, mt['close'], color='#5e81ac', label='收盘价')
ax1.set_ylabel('收盘价', color='#5e81ac')
ax2 = ax1.twinx()
ax2.step(mt.index, mt['ROE'], color='#bf616a', where='post', label='对齐后 ROE(%)')
ax2.set_ylabel('ROE %', color='#bf616a')
ax1.set_title('merge_asof:茅台日线价格叠加按时间就近对齐的季度 ROE')
plt.tight_layout(); plt.savefig('chart_02.png', dpi=110); plt.close()

# 图3:三只票 ROE 阶梯对比
fig, ax = plt.subplots(figsize=(11, 5))
for name in STOCKS.values():
    pf = prof[prof['股票'] == name].sort_values('财报日')
    ax.step(pf['财报日'], pf['ROE'], where='post', marker='o', label=name)
ax.set_title('三只票季度 ROE(%)对比'); ax.set_ylabel('ROE %'); ax.legend()
plt.tight_layout(); plt.savefig('chart_03.png', dpi=110); plt.close()

# 图4:选股结果
fig, ax = plt.subplots(figsize=(8, 5))
pick_count.plot(kind='bar', ax=ax, color='#a3be8c')
ax.set_title('按"当日可见最新 ROE 最高"选股 — 各票入选天数'); ax.set_ylabel('天数')
plt.xticks(rotation=0)
plt.tight_layout(); plt.savefig('chart_04.png', dpi=110); plt.close()

print('\n[done] 4 张图已保存,output.txt 见上方全部打印')

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 数据合并 |  | 把股价和财报用 merge_asof 对齐,做基本面因子选股,是量化最常见的数据准备步骤 |
| 数据合并 |  | 多只票的行情用 concat 竖叠成长表,再 groupby 算横截面因子 |
| 数据合并 |  | 把行情和行业、市值等维表用 merge 贴标签,做行业中性化和分层回测 |
| 数据合并 |  | 宏观指标(利率、汇率)用 merge_asof 对齐到每日策略信号 |
| 数据合并 |  | 券商把成交流水按订单号 merge 客户信息,做交易行为分析 |


## 常见坑

### ⚠ 01. 按行号硬拼不同频率的表

日线和季报行数天差地别,按行号 concat 会让时间完全错位,必须按时间或键对齐。

### ⚠ 02. 用最新财报回测历史

最致命:拿还没披露的财报去买过去的股票,典型未来函数,回测虚高。用 merge_asof backward 杜绝。

### ⚠ 03. merge 后行数暴涨

键有重复时 merge 会产生笛卡尔式的多对多膨胀,合并前先确认键是否唯一。

### ⚠ 04. 忽略 outer join 的 NaN

outer 后冒出的 NaN 是没匹配上的信号,要去查为什么对不上,而不是直接填掉。

### ⚠ 05. merge_asof 忘了先排序

merge_asof 要求两张表都按对齐键排好序,没排序结果会错乱。

## 实战 SOP · 数据合并实战 7 条 SOP

1. 拼表先想清按什么对齐 — 按键用 merge,按位置叠用 concat,按时间就近用 merge_asof。
2. 财报对行情必用 merge_asof backward — 只对齐当天之前已披露的,杜绝未来函数。
3. merge 前确认键唯一 — 防止多对多膨胀导致行数暴涨。
4. 看懂 outer 的 NaN — NaN 是没匹配上的信号,先查原因再决定处理。
5. merge_asof 前先排序 — 两张表都按对齐键排好序。
6. concat 注意索引 — 竖叠时常需 ignore_index 重排,避免重复索引。
7. 对齐后核对样本 — 抽几行看财报日是否都不晚于交易日,确认时点正确。

> 把这段打印贴在你电脑边。

## 总结 · 你应该带走的

2. concat 沿轴拼接:竖着叠加行、横着接加列
3. merge 按键对齐,类 SQL JOIN,inner 取交集、outer 取并集
4. outer join 的 NaN 是"没匹配上"的信号,不是数据缺失
5. merge_asof 给时间序列做就近对齐,backward 只看过去
6. 财报对齐行情必须时点正确,杜绝用未公布数据偷看未来
7. merge 前确认键唯一、merge_asof 前先排序
8. 对齐后抽样核对财报日不晚于交易日

## 自测题

**Q1.** merge 和 concat 的核心区别?(提示:merge 按某个共同的键把两张表对齐拼接(类 SQL JOIN);concat 不看键,只是沿某个方向把表拼起来(竖着叠加行或横着接列)。)

**Q2.** 为什么财报对齐行情必须用 merge_asof backward?(提示:财报有披露滞后,backward 让每个交易日只匹配到当天或之前最近一期已公布的财报,保证时点正确、不会用到未来才披露的数据,杜绝未来函数。)

**Q3.** outer join 之后出现的 NaN 说明什么?(提示:说明这一行在另一张表里没有匹配的键,是"没对上"的信号,提醒你两张表对不齐,应该去查原因,而不是当成普通缺失值直接填掉。)

**Q4.** 小孙把日线和季报按行号直接拼,为什么错?(提示:日线一年两百多行、季报一年才四行,频率完全不同,按行号拼会让某天股价配错季度的财报,时间彻底错位,还可能用到未披露的财报造成未来函数。)

把答案写下来,3 天后再回看。

## 下一节预告

**Day 040 · matplotlib 量化作图** (Matplotlib)

第40 课讲 matplotlib 量化作图:plt 和 ax 两种风格的区别、subplots 多面板布局、twinx 双 y 轴、mdates 日期格式化,带你把净值、回撤、均线信号、成交量画成一张专业的量化分析图。

## 推荐阅读

- Wes McKinney《Python for Data Analysis》(2022, O'Reilly)— merge/concat/join 的权威讲解,含 merge_asof
- pandas 官方文档《Merge, join, concatenate and compare》(2024)— 各种合并方式的标准参考
- Marcos López de Prado《Advances in Financial Machine Learning》(2018, Wiley)— point-in-time 数据与未来函数防范的工业级标准
- Tulchinsky 等《Finding Alphas》(2019, Wiley)— 因子构建中行情与基本面数据对齐的实务
- pandas 官方文档《pandas.merge_asof》(2024)— 时间就近对齐的参数与方向说明